1 - create a file level isntru test review
2 - aggregate the results to create a repo level instru review

In [4]:
import os
import re
import pandas as pd
from typing import List, Pattern, Tuple, Dict, Set

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_8_Test\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

# Optional: allow unreferenced scripts in known CI paths (kept False for minimum false-positives)
LOOSEN_KNOWN_CI_PATHS = False
KNOWN_CI_DIRS = ('.github/scripts', '.github/workflows/scripts', 'ci', 'scripts/ci', 'tools/ci')

# --- Regex helpers ---
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def collect_matches(named_patterns: List[Tuple[str, List[Pattern]]], text: str) -> List[str]:
    hits = []
    for label, pats in named_patterns:
        if any_match(pats, text):
            hits.append(label)
    return hits

# --- Comment scrubbing ---
BLOCK_COMMENT_RE = re.compile(r'/\*.*?\*/', re.S)            # Gradle/KTS block
LINE_COMMENT_RE = re.compile(r'(?m)^\s*//.*?$')              # Gradle/KTS line
YAML_SHELL_LINE_COMMENT_RE = re.compile(r'(?m)^\s*#.*?$')    # YAML/SH/PS1 line

# BAT/CMD: lines starting with REM or :: are comments
BAT_REM_LINE_RE = re.compile(r'(?mi)^\s*(REM\b.*)$')
BAT_COLON_COMMENT_RE = re.compile(r'(?m)^\s*::.*$')

# PS1 block comments: <# ... #>
PS1_BLOCK_COMMENT_RE = re.compile(r'<#.*?#>', re.S | re.I)

def strip_comments(raw: str, ext: str) -> str:
    e = ext.lower()
    txt = raw
    if e in ('.yml', '.yaml', '.sh'):
        txt = YAML_SHELL_LINE_COMMENT_RE.sub('', txt)
        return txt
    if e in ('.gradle', '.kts'):
        txt = BLOCK_COMMENT_RE.sub('', txt)
        txt = LINE_COMMENT_RE.sub('', txt)
        return txt
    if e in ('.bat', '.cmd'):
        txt = BAT_REM_LINE_RE.sub('', txt)
        txt = BAT_COLON_COMMENT_RE.sub('', txt)
        return txt
    if e == '.ps1':
        txt = PS1_BLOCK_COMMENT_RE.sub('', txt)
        txt = YAML_SHELL_LINE_COMMENT_RE.sub('', txt)  # PS1 also uses # for line comments
        return txt
    # .json and others: leave as-is
    return txt

# === DEVICE SETUP sources (comment-safe anchors where applicable) ===
REAL_DEVICE_SOURCES = [
    ('adb devices',       [r'(?m)^(?!\s*#)\s*adb\s+devices\b']),
    ('adb get-state',     [r'(?m)^(?!\s*#)\s*adb\s+get-state\b']),
    ('adb get-serialno',  [r'(?m)^(?!\s*#)\s*adb\s+get-serialno\b']),
    ('adb install',       [r'(?m)^(?!\s*#)\s*adb\s+install(\s+-r)?\b']),
    ('adb -s',            [r'(?m)^(?!\s*#)\s*adb\s+-s\s+\S+\b']),
    ('adb shell',         [r'(?m)^(?!\s*#)\s*adb\s+shell\b']),
    ('adb root',          [r'(?m)^(?!\s*#)\s*adb\s+root\b']),
    ('adb settings',      [r'(?m)^(?!\s*#)\s*adb\s+shell\s+settings\b']),
    ('adb input',         [r'(?m)^(?!\s*#)\s*adb\s+shell\s+input\b']),
    ('adb pm grant',      [r'(?m)^(?!\s*#)\s*adb\s+shell\s+pm\s+grant\b']),
]
EMULATOR_SOURCES = [
    ('sdkmanager/avdmanager', [r'(?m)^(?!\s*#)\s*(sdkmanager|avdmanager)\b']),
    ('emulator -avd/@',       [r'(?m)^(?!\s*#)\s*emulator\s+(-avd|@)\S+']),
    ('android-wait-for-emulator', [r'(?m)^(?!\s*#)\s*android-wait-for-emulator\b']),
    ('start-emulator.sh',     [r'(?m)^(?!\s*#)\s*start-emulator\.sh\b']),
    ('reactivecircus/android-emulator-runner', [r'uses:\s*reactivecircus/android-emulator-runner']),
    ('actions/setup-android', [r'uses:\s*actions/setup-android']),
    ('pierotofy/setup-android', [r'uses:\s*pierotofy/setup-android']),
    ('api-level key',         [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ('ABI/arch key',          [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ('target image',          [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ('device name',           [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),
]
THIRD_PARTY_SOURCES = [
    ('gcloud firebase',       [r'(?m)^(?!\s*#)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ('saucectl',              [r'(?m)^(?!\s*#)\s*saucectl(\s+run)?\b']),
    ('browserstack/bstack',   [r'(?m)^(?!\s*#)\s*(browserstack|bstack)\b']),
    ('appcenter test android',[r'(?m)^(?!\s*#)\s*appcenter\s+test\s+run\s+android\b']),
    ('maestro cloud',         [r'(?m)^(?!\s*#)\s*maestro\s+cloud\b']),
    ('test_matrix.json/firebase.json', [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]
REAL_DEVICE_PATTERNS = [(label, compile_any(pats)) for label, pats in REAL_DEVICE_SOURCES]
EMULATOR_PATTERNS = [(label, compile_any(pats)) for label, pats in EMULATOR_SOURCES]
THIRD_PARTY_PATTERNS = [(label, compile_any(pats)) for label, pats in THIRD_PARTY_SOURCES]

# === TEST TRIGGERS (comment-safe) ===
TRIGGER_PATTERNS = {
    'Gradle': [
        ('gradle connectedAndroidTest',      [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected(android)?test\b']),
        ('gradle connected.*Android.*Test',  [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*android.*test\b']),
        ('gradle connectedCheck',            [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connectedcheck\b']),
        ('gradle createInstrCoverage',       [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*createinstrumentationtestcoveragereport\b']),
        ('gradle runInstrumentationTests',   [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*runinstrumentationtests\b']),
        ('gradle executeScreenshotTests',    [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*executescreenshottests\b']),
        ('gradle orchestrator task',         [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+.*orchestrator\b']),
        ('yaml script -> gradle connected',  [r'\bscript\s*:\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*']),
        ('gradle connected (broad)',         [r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?connected.*\b']),
    ],
    'ADB': [
        ('adb am instrument', [r'(?m)^(?!\s*#)\s*(adb\s+shell\s+)?am\s+instrument\b']),
    ],
    'Third_Party_Lab': [
        ('gcloud firebase',   [r'(?m)^(?!\s*#)\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
        ('saucectl',          [r'(?m)^(?!\s*#)\s*saucectl(\s+run)?\b']),
        ('appcenter test android', [r'(?m)^(?!\s*#)\s*appcenter\s+test\s+run\s+android\b']),
    ],
}
TRIGGER_PATTERNS = {k: [(label, compile_any(pats)) for label, pats in v] for k, v in TRIGGER_PATTERNS.items()}

# === UNIT TESTS (comment-safe) ===
UNIT_TEST_TRIGGER_PATTERNS = compile_any([
    r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?test(?!.*connected)(?!.*android)\b',
    r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?check(?!.*connected)(?!.*android)\b',
    r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?jvmtest\b',
    r'(?m)^(?!\s*#)\s*(\./|\.\\)?gradlew(\.bat)?\s+(:[\w-]+:)?build(?!.*connected)(?!.*android)\b',
    r'(?m)^(?!\s*#)\s*(npm|yarn)\s+test\b',
    r'(?m)^(?!\s*#)\s*run:\s*test\b',
])
UNIT_TEST_CONFIG_PATTERNS = compile_any([
    r'\btestimplementation\b',
    r'\bkotlin\(["\']test["\']\)',
    r'\bandroidTestUtil\b',
    r'\buseOrchestrator\s*=\s*true\b',
    r'junit:junit',
    r'org\.junit\.jupiter',
    r'mockito|mockk\b',
    r'\brobolectric\b',
    r'com\.google\.truth:truth',
    r'org\.hamcrest',
])

# === Platform strictness (unchanged) ===
STRICT_CI_PLATFORMS = ['github_actions', 'gitlab', 'jenkins', 'azure']
LENIENT_CI_PLATFORMS = ['travis_ci', 'bitrise', 'circle_ci', 'appveyor', 'teamcity', 'buddy']

def summarize_device_setup(text: str):
    types, details = set(), []
    for src_type, patterns in [('Real_Device', REAL_DEVICE_PATTERNS),
                               ('Emulator', EMULATOR_PATTERNS),
                               ('Third_Party_Lab', THIRD_PARTY_PATTERNS)]:
        hits = collect_matches(patterns, text)
        if hits:
            types.add(src_type)
            details.extend([f"{src_type}[{h}]" for h in hits])
    return types, details

def summarize_test_triggers(text: str):
    types, details = set(), []
    for ttype, named_pats in TRIGGER_PATTERNS.items():
        hits = collect_matches(named_pats, text)
        if hits:
            types.add(ttype)
            details.extend([f"{ttype}[{h}]" for h in hits])
    return types, details

# === Extract referenced scripts from YAML (Unix & Windows) ===
# We look for explicit calls to scripts or interpreters.
YAML_RUN_LINE_RE = re.compile(
    r'(?mi)^\s*(?:-+\s*)?(run|script)\s*:\s*(?:\|\s*|\>\s*)?(.+)$'
)
# paths or commands inside a run-line chunk
SCRIPT_PATH_RE = re.compile(
    r'(?P<path>'
    r'(\./|\.\\)?[\w\-/\\.]+'
    r'(\.sh|\.bat|\.cmd|\.ps1|\.py|\.js|\.mjs|\.ts)?'  # allow no-ext too; we’ll filter by ext later
    r')\b'
)
# interpreter invocations that include a path argument we can capture
INTERPRETER_CALL_RE = re.compile(
    r'(?mi)\b('
    r'bash|sh|pwsh|powershell|cmd(\.exe)?|call|python|node|npm|yarn|ruby|gradlew(\.bat)?'
    r')\b.*?(?P<path>(\./|\.\\)?[\w\-/\\\.]+(\.sh|\.bat|\.cmd|\.ps1|\.py|\.js|\.mjs|\.ts)?)'
)

def find_referenced_scripts(yaml_text: str) -> Set[str]:
    refs = set()
    for m in YAML_RUN_LINE_RE.finditer(yaml_text):
        chunk = (m.group(2) or '').strip()
        if not chunk:
            continue
        # capture interpreter-style references
        for p in INTERPRETER_CALL_RE.finditer(chunk):
            refs.add(os.path.normpath(p.group('path')))
        # capture direct script paths
        for p in SCRIPT_PATH_RE.finditer(chunk):
            refs.add(os.path.normpath(p.group('path')))
    return refs

def in_known_ci_dir(filename: str) -> bool:
    norm = filename.replace('\\', '/').lower()
    return any(norm.startswith(d.lower() + '/') or ('/' + d.lower() + '/') in norm for d in KNOWN_CI_DIRS)

# === First pass: read and pre-process all files ===
files = []
for file in os.listdir(CONFIG_DIR):
    file_path = os.path.join(CONFIG_DIR, file)
    if not os.path.isfile(file_path):
        continue
    ext = os.path.splitext(file)[-1].lower()
    if ext not in ['.sh', '.json', '.gradle', '.kts', '.yml', '.yaml', '.bat', '.cmd', '.ps1']:
        continue
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()
    cleaned = strip_comments(raw, ext)
    files.append({
        'name': file,
        'path': file_path,
        'ext': ext,
        'content': cleaned,
    })

# === Gate discovery (YAML only) ===
instrumentation_gate = False
referenced_scripts_global: Set[str] = set()
yaml_results_temp: Dict[str, Dict] = {}

for f in files:
    if f['ext'] not in ('.yml', '.yaml'):
        continue

    file = f['name']
    content = f['content'].lower()

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    ds_types, ds_detail = summarize_device_setup(content)
    trig_types, trig_detail = summarize_test_triggers(content)

    if ds_types or trig_types:
        instrumentation_gate = True

    refs = find_referenced_scripts(content)
    referenced_scripts_global |= refs

    yaml_results_temp[file] = {
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup_types': ds_types,
        'device_setup_detail': ds_detail,
        'trigger_types': trig_types,
        'trigger_detail': trig_detail,
        'parsed_ok': True
    }

# === Second pass: produce final rows with strict linking & gating ===
results = []

for f in files:
    file = f['name']; ext = f['ext']; content = f['content'].lower()

    full_name, ci_platform = "Unknown", "Unknown"
    if "__" in file and "++" in file:
        try:
            full_name = file.split("__")[0]
            ci_platform = file.split("__")[1].split("++")[0]
        except Exception:
            pass

    parsed_ok = True
    file_is_build = ext in ['.gradle', '.kts']
    file_is_yaml = ext in ['.yml', '.yaml']

    device_setup_types, device_setup_detail = set(), []
    trigger_types, trigger_detail = set(), []
    unit_test_trigger = False
    unit_test_config = False
    instru_test_status = "None"

    try:
        if file_is_yaml:
            y = yaml_results_temp.get(file, None)
            if y:
                device_setup_types = y['device_setup_types']
                device_setup_detail = y['device_setup_detail']
                trigger_types = y['trigger_types']
                trigger_detail = y['trigger_detail']

                platform_key = (y['ci_platform'] or '').strip().lower()
                ds = "None" if not device_setup_types else ", ".join(sorted(device_setup_types))
                ht = bool(trigger_types)
                if ds == "None" and ht:
                    instru_test_status = "Defective" if platform_key in STRICT_CI_PLATFORMS else "Complete"
                elif ds != "None" and ht:
                    instru_test_status = "Complete"
                elif ds != "None" and not ht:
                    instru_test_status = "Manual"

        else:
            # SUPPORTING FILES: only if gate is open AND (strictly referenced OR in known CI dir when loosened)
            if instrumentation_gate:
                normfile = os.path.normpath('./' + file)
                is_referenced = (file in referenced_scripts_global) or (normfile in referenced_scripts_global)
                allowed_by_dir = LOOSEN_KNOWN_CI_PATHS and in_known_ci_dir(file)

                if is_referenced or allowed_by_dir:
                    # Detect device setup & triggers inside the script
                    if ext in ('.sh', '.ps1', '.bat', '.cmd', '.json'):
                        ds, dsdet = summarize_device_setup(content)
                        tg, tgdet = summarize_test_triggers(content)
                        device_setup_types |= ds
                        device_setup_detail.extend(dsdet)
                        trigger_types |= tg
                        trigger_detail.extend(tgdet)
                        if any_match(UNIT_TEST_TRIGGER_PATTERNS, content):
                            unit_test_trigger = True

                    if file_is_build:
                        if any_match(UNIT_TEST_CONFIG_PATTERNS, content):
                            unit_test_config = True
            # else: ignore—keeps false positives near zero

    except Exception:
        parsed_ok = False
        device_setup_types, device_setup_detail = set(), []
        trigger_types, trigger_detail = set(), []
        unit_test_trigger = False
        unit_test_config = False
        instru_test_status = "Error while parsing"

    results.append({
        'filename': file,
        'file_type': ext[1:],
        'full_name': full_name,
        'ci_platform': ci_platform,
        'device_setup': "None" if not device_setup_types else ", ".join(sorted(device_setup_types)),
        'device_setup_detail': "; ".join(device_setup_detail) if device_setup_detail else "",
        'has_device_setup': bool(device_setup_types),
        'has_test_trigger': bool(trigger_types),
        'test_trigger': ", ".join(sorted(trigger_types)) if trigger_types else "",
        'test_trigger_detail': "; ".join(trigger_detail) if trigger_detail else "",
        'unit_test_trigger': unit_test_trigger,
        'unit_test_config': unit_test_config,
        'instru_test_status': instru_test_status,
        'parsed_ok': parsed_ok
    })

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Instrumentation analysis complete (YAML-gated, strict-linked; .bat/.cmd supported). Saved to:\n{OUTPUT_CSV}")


✅ Instrumentation analysis complete (YAML-gated, strict-linked; .bat/.cmd supported). Saved to:
C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_Config_Files_List_ShallowC.csv
